In [1]:
import duckdb

con = duckdb.connect("../dbt_project/warehouse.duckdb")
con.sql("show tables").fetchall()

[('dim_station',),
 ('dim_tract',),
 ('fact_lcdv2_tract_hourly',),
 ('int_lcdv2_hourly',),
 ('int_lcdv2_hourly_weighted',),
 ('int_lcdv2_hourly_weighted_bucket_00',),
 ('int_lcdv2_hourly_weighted_bucket_01',),
 ('int_lcdv2_hourly_weighted_bucket_02',),
 ('int_lcdv2_hourly_weighted_bucket_03',),
 ('int_lcdv2_hourly_weighted_bucket_04',),
 ('int_lcdv2_hourly_weighted_bucket_05',),
 ('int_lcdv2_hourly_weighted_bucket_06',),
 ('int_lcdv2_hourly_weighted_bucket_07',),
 ('int_lcdv2_hourly_weighted_bucket_08',),
 ('int_lcdv2_hourly_weighted_bucket_09',),
 ('int_lcdv2_hourly_weighted_bucket_10',),
 ('int_lcdv2_hourly_weighted_bucket_11',),
 ('int_lcdv2_hourly_weighted_bucket_12',),
 ('int_lcdv2_hourly_weighted_bucket_13',),
 ('int_lcdv2_hourly_weighted_bucket_14',),
 ('int_lcdv2_hourly_weighted_bucket_15',),
 ('int_prevailing_wind_direction',),
 ('int_station_dim_enriched',),
 ('int_station_hourly',),
 ('int_station_weights',),
 ('int_tract_dim_enriched',),
 ('int_tract_station_distance',),
 (

In [2]:
tables = [
    "dim_station",
    "dim_tract",
    "rel_tract_station_distance",
    "rel_station_weights",
    "fact_lcdv2_tract_hourly"
]

for t in tables:
    print(t, con.sql(f"select count(*) from {t}").fetchone()[0])

dim_station 64
dim_tract 1542
rel_tract_station_distance 98688
rel_station_weights 24756
fact_lcdv2_tract_hourly 22266651


In [3]:
con.sql("describe dim_station")

┌───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │ column_type │  null   │   key   │ default │  extra  │
│    varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ station       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ name          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ latitude      │ FLOAT       │ YES     │ NULL    │ NULL    │ NULL    │
│ longitude     │ FLOAT       │ YES     │ NULL    │ NULL    │ NULL    │
│ elevation_m   │ FLOAT       │ YES     │ NULL    │ NULL    │ NULL    │
│ data_coverage │ FLOAT       │ YES     │ NULL    │ NULL    │ NULL    │
│ min_date      │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ max_date      │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
└───────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [4]:
con.sql("select * from dim_station limit 5")

┌────────────┬────────────────────────────────────────────────┬──────────┬───────────┬─────────────┬───────────────┬────────────┬────────────┐
│  station   │                      name                      │ latitude │ longitude │ elevation_m │ data_coverage │  min_date  │  max_date  │
│  varchar   │                    varchar                     │  float   │   float   │    float    │     float     │    date    │    date    │
├────────────┼────────────────────────────────────────────────┼──────────┼───────────┼─────────────┼───────────────┼────────────┼────────────┤
│ WBAN:00132 │ WAUTOMA MUNICIPAL AIRPORT, WI US               │   44.033 │     -89.3 │    260.3973 │           1.0 │ 2014-07-31 │ 2025-08-25 │
│ WBAN:00183 │ PLATTEVILLE MUNICIPAL AIRPORT, WI US           │   42.683 │    -90.45 │    294.5592 │           1.0 │ 2009-01-01 │ 2025-08-25 │
│ WBAN:00185 │ SHAWANO MUNICIPAL AIRPORT, WI US               │   44.783 │    -88.55 │   247.75769 │           1.0 │ 2009-02-01 │ 2025-08-25 │

In [5]:
con.sql("describe dim_tract")

┌──────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name      │ column_type │  null   │   key   │ default │  extra  │
│       varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ tract                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ state_fips           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ county_fips          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ tract_code           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ tract_bucket         │ UBIGINT     │ YES     │ NULL    │ NULL    │ NULL    │
│ population_total     │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ population_male      │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ population_female    │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ male_65_66           │ INTEGER     │ YES     │ NUL

In [6]:
con.sql("select * from dim_tract limit 5")

┌─────────────┬────────────┬─────────────┬────────────┬──────────────┬──────────────────┬─────────────────┬───────────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬──────────────┬──────────────┬──────────────┬──────────────┬──────────────┬──────────────┬────────────────┬──────────────┬────────────────┬────────────┬─────────────────────┬──────────────────┬────────────────────┬─────────────────┬───────────────┬─────────────┬──────────────────┬───────────────┬──────────────────┬───────────────┬─────────────────────────┬─────────────┬──────────────────────┬───────────────────────┬───────────────┬──────────────────┬────────────────┬───────────────────┬───────────────────┬───────────────┬────────────┬────────────┬──────────────────────┬────────────┬───────────────────────┬───────────────┬────────────┬───────────────────┬──────────────────────┬───────────────────────┬─────────────────────────┬─────────────────┬──────────────────┬────────────────────┬───────────────

In [7]:
con.sql("describe rel_tract_station_distance")

┌──────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name    │ column_type │  null   │   key   │ default │  extra  │
│     varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ station          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ tract            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ distance_m       │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ bearing_deg      │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ elevation_diff_m │ FLOAT       │ YES     │ NULL    │ NULL    │ NULL    │
└──────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [8]:
con.sql("select * from rel_tract_station_distance limit 5")

┌────────────┬─────────────┬────────────────────┬────────────────────┬──────────────────┐
│  station   │    tract    │     distance_m     │    bearing_deg     │ elevation_diff_m │
│  varchar   │   varchar   │       double       │       double       │      float       │
├────────────┼─────────────┼────────────────────┼────────────────────┼──────────────────┤
│ WBAN:00132 │ 55001950100 │   40785.4792966501 │  297.6379093292176 │         51.46814 │
│ WBAN:00132 │ 55001950201 │   46912.4462180845 │ 266.67224780360186 │        22.847412 │
│ WBAN:00132 │ 55001950203 │  35615.20384824882 │  285.7708954960312 │        48.807037 │
│ WBAN:00132 │ 55001950204 │  40196.89374346198 │  276.5066276450684 │        40.023804 │
│ WBAN:00132 │ 55001950400 │ 37031.351708808754 │  254.0402683121767 │        42.596283 │
└────────────┴─────────────┴────────────────────┴────────────────────┴──────────────────┘

In [9]:
con.sql("describe rel_station_weights")

┌───────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │ column_type │  null   │   key   │ default │  extra  │
│      varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ station           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ tract             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ season            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ wind_alignment    │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ elevation_penalty │ FLOAT       │ YES     │ NULL    │ NULL    │ NULL    │
│ distance_factor   │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ raw_weight        │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ final_weight      │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ stations_used     │ VARCHAR[]   │ YES     │ NULL    │ NULL    │ NULL    │
└───────────

In [10]:
con.sql("select * from rel_station_weights limit 5")

┌────────────┬─────────────┬─────────┬──────────────────────┬───────────────────┬─────────────────────┬────────────────────┬─────────────────────┬──────────────────────────────────────────────────────────┐
│  station   │    tract    │ season  │    wind_alignment    │ elevation_penalty │   distance_factor   │     raw_weight     │    final_weight     │                      stations_used                       │
│  varchar   │   varchar   │ varchar │        double        │       float       │       double        │       double       │       double        │                        varchar[]                         │
├────────────┼─────────────┼─────────┼──────────────────────┼───────────────────┼─────────────────────┼────────────────────┼─────────────────────┼──────────────────────────────────────────────────────────┤
│ WBAN:00185 │ 55087940000 │ JJA     │ 0.051007935890685474 │         0.8569506 │ 0.16411314636899843 │ 0.3606538557325959 │   0.178534691277207 │ ['WBAN:00185', 'WBAN:04825', 

In [11]:
con.sql("describe fact_lcdv2_tract_hourly")

┌──────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│         column_name          │ column_type │  null   │   key   │ default │  extra  │
│           varchar            │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ tract                        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ floored_timestamp            │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ stations_used                │ VARCHAR[]   │ YES     │ NULL    │ NULL    │ NULL    │
│ weighted_dry_bulb_temp       │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ weighted_wet_bulb_temp       │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ weighted_dew_point_temp      │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ weighted_relative_humidity   │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ weighted_wind_speed          │ DOUBLE    

In [12]:
con.sql("select * from fact_lcdv2_tract_hourly limit 5")

┌─────────────┬─────────────────────┬────────────────────────────────────────────────────────────────────────┬────────────────────────┬────────────────────────┬─────────────────────────┬────────────────────────────┬─────────────────────┬──────────────────────────┬────────────────────────┬─────────────────────┬───────────────────────────┬──────────────────────────────┬─────────────────────────┐
│    tract    │  floored_timestamp  │                             stations_used                              │ weighted_dry_bulb_temp │ weighted_wet_bulb_temp │ weighted_dew_point_temp │ weighted_relative_humidity │ weighted_wind_speed │ weighted_wind_gust_speed │ weighted_precipitation │ weighted_visibility │ weighted_station_pressure │ weighted_barometric_pressure │ weighted_wind_direction │
│   varchar   │      timestamp      │                               varchar[]                                │         double         │         double         │         double          │           double   

In [13]:
con.close()